In [ ]:
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from pathlib import Path

# 입력 및 출력 경로 설정
in_fp = Path(r"D:\LiDAR\TX_South_B1_2018\merged_TX_South_B1_2018.tif")
out_fp = in_fp.with_name("reprojected_TX_South_B1_2018_epsg4326.tif")

# 타겟 CRS
dst_crs = "EPSG:4326"

with rasterio.open(in_fp) as src:
    # 변환 정보 계산
    transform, width, height = calculate_default_transform(
        src.crs, dst_crs, src.width, src.height, *src.bounds
    )
    
    # 출력 메타데이터 구성
    kwargs = src.meta.copy()
    kwargs.update({
        "crs": dst_crs,
        "transform": transform,
        "width": width,
        "height": height
    })

    # 재투영 및 저장
    with rasterio.open(out_fp, "w", **kwargs) as dst:
        for i in range(1, src.count + 1):
            reproject(
                source=rasterio.band(src, i),
                destination=rasterio.band(dst, i),
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=dst_crs,
                resampling=Resampling.bilinear  # 또는 nearest, cubic 등
            )

print(f"✅ 재투영 완료: {out_fp}")